# XGBoost training

Build a unified training frame with `build_training_frame()`: static IEEE columns from `train_features.parquet`, operational columns from PostgreSQL, and rolling behavioral features computed on the fly.

In [1]:
from fraud_scoring_engine.db.engine import create_db_engine
from fraud_scoring_engine.training import build_training_frame
from sqlalchemy.orm import Session

In [ ]:
TARGET = "is_fraud"
LIMIT = 10_000

In [2]:
engine = create_db_engine()
with Session(engine) as session:
    df = build_training_frame(session, limit=LIMIT)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Postgres columns present: {sum(col in df.columns for col in ['transaction_amt', 'card1', 'product_cd'])} / 3")

TRAIN_FEATURES_PATH: c:\Repos\fraud-scoring-engine\data\processed\train_features.parquet
Exists: True
BEHAVIORAL_FEATURES_PATH: c:\Repos\fraud-scoring-engine\data\processed\behavioral_features.parquet
Exists: True


In [ ]:
display(df.shape)
display(df[TARGET].value_counts())
display(df[["transaction_id", "transaction_amt", "card1", "product_cd", "velocity_1h", TARGET]].head())

(10000, 419)

is_fraud
0    9735
1     265
Name: count, dtype: int64

,transaction_id,C1,C2,C3,C4,C5,C6,C7,C8,C9,...,id_34,id_35,id_36,id_37,id_38,velocity_1h,cumulative_spend_24h,avg_amount_ratio_30d,avg_amount_ratio_90d,is_fraud
0,2987000,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN,0
1,2987001,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN,0
2,2987002,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN,0
3,2987003,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,0,0.0,NaN,NaN,0
4,2987004,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,match_status:2,T,F,T,T,0,0.0,NaN,NaN,0


KeyError: 'card1'

## Feature preprocessing

Use `FraudFeaturePreprocessor` to drop metadata columns, keep numeric nulls for XGBoost, and encode categoricals with a stable `__MISSING__` level.

In [ ]:
from fraud_scoring_engine.preprocessing import FraudFeaturePreprocessor

preprocessor = FraudFeaturePreprocessor()
X = preprocessor.fit_transform(df)
y = df[TARGET]

print(X.shape)
print(X.select_dtypes("category").shape[1])
print(X.select_dtypes("float").shape[1]) 
print(X.select_dtypes("int").shape[1])

(10000, 417)
22
394
1
